In [ ]:
import os
while not os.path.exists("config/config.yaml") and os.getcwd() != os.path.dirname(os.getcwd()):
    os.chdir("../")
os.getcwd()

In [4]:
os.makedirs("artifacts/data_ingestion", exist_ok=True)

In [5]:
dataset_id = "nazmul0087/ct-kidney-dataset-normal-cyst-tumor-and-stone"
os.system(f"kaggle datasets download -d {dataset_id} -p artifacts/data_ingestion --unzip")

Dataset URL: https://www.kaggle.com/datasets/nazmul0087/ct-kidney-dataset-normal-cyst-tumor-and-stone
License(s): Attribution 4.0 International (CC BY 4.0)


100%|██████████| 1.52G/1.52G [01:19<00:00, 20.4MB/s]


0

In [6]:
from dataclasses import dataclass
from pathlib import Path

# Entity / return type for the ConfigurationManager.
# A frozen dataclass that defines what a data ingestion config looks like.
# Frozen means fields are read-only after creation — prevents accidental changes.
# The ConfigurationManager reads config.yaml and returns an instance of this class,
# which is then passed to the DataIngestion component to do the actual work.
@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path        
    source_URL: str       
    local_data_file: Path 
    unzip_dir: Path       

In [7]:
os.listdir("artifacts/data_ingestion")

['kidneyData.csv', 'data.zip', 'CT-KIDNEY-DATASET-Normal-Cyst-Tumor-Stone']

In [8]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories

In [9]:
class ConfigurationManager:
    def __init__(self, config_filepath=CONFIG_FILE_PATH, params_filepath=PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir,
        )

        return data_ingestion_config

In [10]:
import os
import subprocess
import zipfile
from pathlib import Path
from cnnClassifier import logger
from cnnClassifier.utils.common import read_yaml, create_directories

In [11]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    def download_file(self):
        """
        Fetch data from Kaggle
        """
        try:
            dataset_id = self.config.source_URL
            root_dir = Path(self.config.root_dir)
            zip_download_path = Path(self.config.local_data_file)
            os.makedirs(root_dir, exist_ok=True)

            logger.info(f"Downloading data from Kaggle dataset {dataset_id} into {root_dir}")

            subprocess.run(
                ["kaggle", "datasets", "download", "-d", dataset_id, "-p", str(root_dir)],
                check=True,
            )

            downloaded_zip = next(root_dir.glob("*.zip"))
            if downloaded_zip != zip_download_path:
                downloaded_zip.rename(zip_download_path)

            logger.info(f"Downloaded data from Kaggle dataset {dataset_id} into file {zip_download_path}")

        except Exception as e:
            raise e

    def extract_zip_file(self):
        """
        zip_file_path: str
        Extracts the zip file into the data directory
        Function returns None
        """
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)

In [12]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e

[2026-07-01 21:16:02,475: INFO: common]: yaml file: config/config.yaml loaded successfully
[2026-07-01 21:16:02,479: INFO: common]: yaml file: params.yaml loaded successfully
[2026-07-01 21:16:02,479: INFO: common]: created directory at: artifacts
[2026-07-01 21:16:02,480: INFO: common]: created directory at: artifacts/data_ingestion
[2026-07-01 21:16:02,481: INFO: 1112111481]: Downloading data from Kaggle dataset nazmul0087/ct-kidney-dataset-normal-cyst-tumor-and-stone into artifacts/data_ingestion
Dataset URL: https://www.kaggle.com/datasets/nazmul0087/ct-kidney-dataset-normal-cyst-tumor-and-stone
License(s): Attribution 4.0 International (CC BY 4.0)


100%|█████████▉| 1.51G/1.52G [01:22<00:00, 16.6MB/s]


[2026-07-01 21:17:25,982: INFO: 1112111481]: Downloaded data from Kaggle dataset nazmul0087/ct-kidney-dataset-normal-cyst-tumor-and-stone into file artifacts/data_ingestion/data.zip


100%|██████████| 1.52G/1.52G [01:22<00:00, 19.8MB/s]
